In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from collections import defaultdict
import numpy as np
from torch_geometric.utils import softmax

class Generator(nn.Module):
    def __init__(self, num_nodes, embedding_dim, relations):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.relation_emb = nn.Embedding(len(relations), embedding_dim)
        self.relations = relations
        self.bfs_trees = defaultdict(lambda: defaultdict(dict))

    def build_bfs_trees(self, data, walk_length=num):
        for rel in self.relations:
            edge_list = data[rel].edge_index.t().tolist()
            adj_list = defaultdict(list)
            for src, dst in edge_list:
                adj_list[src].append(dst)

            for node in range(data['node'].num_nodes):
                tree = {}
                visited = {node: 0}
                queue = [node]
                while queue and len(visited) < walk_length:
                    current = queue.pop(0)
                    for neighbor in adj_list.get(current, []):
                        if neighbor not in visited:
                            visited[neighbor] = visited[current] + 1
                            tree[neighbor] = current
                            queue.append(neighbor)
                self.bfs_trees[rel][node] = tree

    def forward(self, v_c, rel_idx):
        current = v_c.item() if isinstance(v_c, torch.Tensor) else v_c
        rel = self.relations[rel_idx]

        path = [current]
        for _ in range(num):
            neighbors = list(self.bfs_trees[rel][current].keys())
            if not neighbors:
                break

            rel_emb = self.relation_emb(torch.tensor([rel_idx]))
            current_emb = self.node_emb(torch.tensor([current])) + rel_emb
            neighbor_embs = self.node_emb(torch.tensor(neighbors)) + rel_emb

            logits = torch.mm(neighbor_embs, current_emb.T).squeeze()
            probs = F.softmax(logits, dim=num)

            if torch.rand(1) < 0.1:
                next_node = np.random.choice(neighbors)
            else:
                next_node = neighbors[probs.argmax()]

            path.append(next_node)
            current = next_node

        return current

class Discriminator(nn.Module):
    def __init__(self, num_nodes, embedding_dim, relations):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.relation_emb = nn.Embedding(len(relations), embedding_dim)
        self.relations = relations

    def forward(self, src, dst, rel_idx):
        rel_emb = self.relation_emb(rel_idx)
        src_emb = self.node_emb(src) * rel_emb
        dst_emb = self.node_emb(dst) * rel_emb
        return torch.sigmoid(torch.sum(src_emb * dst_emb, dim=-1))

class HeteroEdgePruner:
    def __init__(self, discriminator, thresholds):
        self.discriminator = discriminator
        self.thresholds = thresholds

    def prune_edges(self, data, train_mask):
        pruned_data = HeteroData()
        pruned_data['node'].x = data['node'].x
        pruned_data['node'].y = data['node'].y

        for rel_idx, rel in enumerate(data.edge_types):
            edge_index = data[rel].edge_index
            with torch.no_grad():
                scores = self.discriminator(
                    edge_index[0],
                    edge_index[1],
                    torch.full((edge_index.size(1),), rel_idx, device=edge_index.device)
                )
            mask = (scores > self.thresholds[rel_idx]) | (train_mask[edge_index[0]])
            pruned_data[rel].edge_index = edge_index[:, mask]

        return pruned_data

class CovarianceDisentangler(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_relations):
        super().__init__()
        self.num_relations = num_relations
        self.base_encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LeakyReLU(num),
            nn.LayerNorm(hidden_dim))

        self.relation_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.Tanh()
            ) for _ in range(num_relations)
        ])

    def forward(self, x):
        base_feat = self.base_encoder(x)
        relation_feats = [head(base_feat) for head in self.relation_heads]
        return base_feat, relation_feats

    def covariance_loss(self, relation_feats):
        loss = 0
        for feat in relation_feats:
            if feat.size(0) < 2:
                continue
            feat_t = feat.T
            cov = torch.cov(feat_t) + 1e-4 * torch.eye(feat_t.size(0), device=feat.device)
            identity = torch.eye(cov.size(0), device=feat.device)
            loss += torch.norm(cov - identity, p='fro') / feat.size(0)
        return loss / max(1, self.num_relations)

class RelationInternalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__(aggr='add', node_dim=0)
        self.attn = nn.Parameter(torch.Tensor(1, 2 * hidden_dim))
        self.gate = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        self.lin = nn.Linear(hidden_dim, hidden_dim)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.attn)
        self.gate[0].reset_parameters()

    def forward(self, x, edge_index):
        x = self.lin(x)
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j, edge_index):
        alpha = (torch.cat([x_i, x_j], dim=-1) * self.attn).sum(dim=-1)
        alpha = F.leaky_relu(alpha, 0.2)
        alpha = softmax(alpha, edge_index[0])

        gate_input = torch.cat([x_i, x_j, x_i - x_j], dim=-1)
        gate = self.gate(gate_input)

        return x_j * alpha.unsqueeze(-1) * gate

class RelationLevelAttention(nn.Module):
    def __init__(self, hidden_dim, num_relations):
        super().__init__()
        self.relation_emb = nn.Embedding(num_relations, hidden_dim)

        self.gate_net = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        self.attn_net = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, base_feat, relation_feats):
        weights = []
        for i, rel_feat in enumerate(relation_feats):
            rel_emb = self.relation_emb(torch.tensor([i], device=base_feat.device))
            expanded_rel_emb = rel_emb.expand_as(rel_feat)
            gate_input = torch.cat([expanded_rel_emb, rel_feat], dim=-1)
            gate = self.gate_net(gate_input)
            fused_relation = gate * expanded_rel_emb + (1 - gate) * rel_feat
            combined = torch.cat([base_feat, fused_relation], dim=-1)
            weight = self.attn_net(combined)
            weights.append(weight)

        weights = torch.stack(weights, dim=1)
        return F.softmax(weights, dim=1)

class HierarchicalAggregator(nn.Module):
    def __init__(self, hidden_dim, relations):
        super().__init__()
        self.relations = relations
        self.num_relations = len(relations)
        self.internal_attentions = nn.ModuleDict({
            '_'.join(rel): RelationInternalAttention(hidden_dim) for rel in relations
        })
        self.relation_attention = RelationLevelAttention(hidden_dim, self.num_relations)

    def forward(self, base_feat, relation_feats, edge_indices, batch_nodes=None):
        intra_aggregated = []
        for i, rel in enumerate(self.relations):
            edge_index = edge_indices[rel]
            if batch_nodes is not None:
                batch_tensor = torch.tensor(batch_nodes, device=edge_index.device)
                mask = torch.isin(edge_index[1], batch_tensor)
                edge_index = edge_index[:, mask]

            if edge_index.size(1) == 0:
                intra_aggregated.append(relation_feats[i])
                continue

            agg_feat = self.internal_attentions['_'.join(rel)](
                relation_feats[i],
                edge_index
            )
            intra_aggregated.append(agg_feat)
        relation_weights = self.relation_attention(
            base_feat,
            intra_aggregated,
            torch.arange(self.num_relations, device=base_feat.device)
        )

        fused_feat = torch.stack(intra_aggregated, dim=1)
        fused_feat = (fused_feat * relation_weights).sum(dim=1)
        return fused_feat

class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, relations):
        super().__init__()
        self.relations = relations
        self.disentangler = CovarianceDisentangler(in_dim, hidden_dim, len(relations))
        self.aggregator = HierarchicalAggregator(hidden_dim, relations)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(num),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data, batch_nodes=None):
        base_feat, relation_feats = self.disentangler(data['node'].x)
        edge_indices = {rel: data[rel].edge_index for rel in self.relations}

        fused_feat = self.aggregator(base_feat, relation_feats, edge_indices, batch_nodes)
        combined = torch.cat([base_feat, fused_feat], dim=1)

        if batch_nodes is not None:
            return F.log_softmax(self.classifier(combined[batch_nodes]), dim=1)
        return F.log_softmax(self.classifier(combined), dim=1)